# Preprocessing on the IMDB Movie Reviews Dataset

The dataset contains `review` and `sentiment` columns.

The steps covered in this preprocessing pipeline are:
1. Loading the dataset
2. Cleaning text (removing HTML tags, URLs, numbers, etc.) using Regular Expressions
3. Expanding contractions
4. Lowercasing and removing punctuation
5. Tokenization using NLTK
6. Stop-word removal using NLTK
7. Lemmatization using spaCy
8. Applying the pipeline to the dataset


## Import Libraries

In [1]:
import pandas as pd
import re
import contractions
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  #tokenizer + lemmatizer


## 1. Load and Explore Dataset

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

print("Shape of dataset:", df.shape)
print("\nColumn names:", list(df.columns))
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

df.head()

Shape of dataset: (50000, 2)

Column names: ['review', 'sentiment']

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


at a raw review to understand what needs cleaning:

In [3]:
sample_review = df.loc[0, 'review']
print(sample_review)

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

For testing purposes, we use a sample of 1000 reviews.

In [4]:
sample = df.sample(1000, random_state=42).reset_index(drop=True)
sample.head()

,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative


## 2. Clean Text (RegEx)

Raw reviews often contain HTML tags (like `<br />`), URLs, and numbers. We use Regular Expressions to remove these.

In [5]:
def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text)                  # remove HTML tags like <br />
    text = re.sub(r'http\S+|www\S+', ' ', text)         # remove URLs
    text = re.sub(r'\d+', ' ', text)                     # remove standalone numbers
    text = text.lower()                                  # lowercase everything
    text = re.sub(r'\s+', ' ', text).strip()             # collapse multiple spaces
    return text

# Test on our sample review
cleaned_example = clean_text(sample_review)
print("BEFORE:\n", sample_review[:300])
print("\nAFTER CLEANING:\n", cleaned_example[:300])


BEFORE:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

AFTER CLEANING:
 one of the other reviewers has mentioned that after watching just oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this i


## 3. Expand Contractions

We expand contractions (e.g., `don't` to `do not`) before removing punctuation.

In [6]:
def expand_contractions(text):
    return contractions.fix(text)

expanded_example = expand_contractions(cleaned_example)
print("BEFORE:\n", cleaned_example[:300])
print("\nAFTER EXPANDING CONTRACTIONS:\n", expanded_example[:300])


BEFORE:
 one of the other reviewers has mentioned that after watching just oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this i

AFTER EXPANDING CONTRACTIONS:
 one of the other reviewers has mentioned that after watching just oz episode you will be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this


## 4. Remove Punctuation

Removing special characters and punctuation to keep only alphabetic characters.

In [7]:
def remove_punctuation(text):
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep only lowercase letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

no_punct_example = remove_punctuation(expanded_example)
print("AFTER REMOVING PUNCTUATION:\n", no_punct_example[:300])


AFTER REMOVING PUNCTUATION:
 one of the other reviewers has mentioned that after watching just oz episode you will be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is no


## 5. Tokenization

Splitting the text into individual words using NLTK.

In [8]:
tokens_example = word_tokenize(no_punct_example)
print("TOKENS:\n", tokens_example[:20])
print("\nTotal tokens:", len(tokens_example))


TOKENS:
 ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'will', 'be', 'hooked', 'they', 'are', 'right']

Total tokens: 314


## 6. Stop-word Removal

Removing common stop-words using NLTK.

In [9]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [w for w in tokens if w not in stop_words]

filtered_example = remove_stopwords(tokens_example)
print("AFTER STOPWORD REMOVAL:\n", filtered_example[:20])
print("\nTotal tokens after stopword removal:", len(filtered_example))


AFTER STOPWORD REMOVAL:
 ['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right']

Total tokens after stopword removal: 162


## 7. Lemmatization and Stemming

Comparing NLTK's Stemmer with spaCy's Lemmatizer. We will use lemmatization in our final pipeline.

In [10]:
stemmer = PorterStemmer()
stemmed_example = [stemmer.stem(w) for w in filtered_example]

doc = nlp(" ".join(filtered_example))
lemmatized_example = [token.lemma_ for token in doc]

comparison = pd.DataFrame({
    "original": filtered_example[:15],
    "stemmed (NLTK)": stemmed_example[:15],
    "lemmatized (spaCy)": lemmatized_example[:15]
})
comparison


,original,stemmed (NLTK),lemmatized (spaCy)
0,one,one,one
1,reviewers,review,reviewer
2,mentioned,mention,mention
3,watching,watch,watch
4,oz,oz,oz
5,episode,episod,episode
6,hooked,hook,hook
7,right,right,right
8,exactly,exactli,exactly
9,happened,happen,happen


## 8. Complete Preprocessing Pipeline

Combining all the steps into a single preprocessing function.

In [11]:
def preprocess_text(text):
    text = clean_text(text)
    text = expand_contractions(text)
    text = remove_punctuation(text)
    tokens = word_tokenize(text)
    tokens = remove_stopwords(tokens)
    doc = nlp(" ".join(tokens))
    lemmas = [token.lemma_ for token in doc]
    return " ".join(lemmas)

# Quick test
print(preprocess_text(sample_review))


one reviewer mention watch oz episode hook right exactly happen first thing strike oz brutality unflinche scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use word call oz nickname give oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inward privacy high agenda city home many aryans muslims gangstas latinos christians italian irish scuffle death stare dodgy dealing shady agreement never far away would say main appeal show due fact go show would dare forget pretty picture paint mainstream audience forget charm forget romance oz mess around first episode ever see strike nasty surreal could say ready watch develop taste oz get accustomed high level graphic violence violence injustice crook guard sell nickel inmate kill order get away well mannered middle class inmate turn prison bitch due lack street skill prison experience watch oz may become comfo

Applying the preprocessing function to our dataset.

In [12]:
sample['cleaned_review'] = sample['review'].apply(preprocess_text)
sample[['review', 'cleaned_review', 'sentiment']].head(10)


,review,cleaned_review,sentiment
0,I really liked this Summerslam due to the look...,really like summerslam due look arena curtain ...,positive
1,Not many television shows appeal to quite as m...,many television show appeal quite many differe...,positive
2,The film quickly gets to a major chase scene w...,film quickly get major chase scene ever increa...,negative
3,Jane Austen would definitely approve of this o...,jane austen would definitely approve one gwyne...,positive
4,Expectations were somewhat high for me when I ...,expectation somewhat high go see movie think s...,negative
5,I've watched this movie on a fairly regular ba...,watch movie fairly regular basis life never ge...,positive
6,For once a story of hope highlighted over the ...,story hope highlight tragic reality youth face...,positive
7,"Okay, I didn't get the Purgatory thing the fir...",okay get purgatory thing first time watch epis...,positive
8,I was very disappointed with this series. It h...,disappoint series lot cool graphic level detai...,negative
9,The first 30 minutes of Tinseltown had my fing...,first minute tinseltown finger teeter remote p...,negative


## 9. Save Preprocessed Data

In [13]:
sample[['review', 'cleaned_review', 'sentiment']].to_csv("IMDB_Dataset_preprocessed_sample.csv", index=False)
print("Saved: IMDB_Dataset_preprocessed_sample.csv")


Saved: IMDB_Dataset_preprocessed_sample.csv
